In [1]:
import numpy as np
import random

In [3]:
distance_matrix = np.array([
    [0, 2, 9, 10, 7],
    [2, 0, 6, 4, 3],
    [9, 6, 0, 8, 5],
    [10, 4, 8, 0, 6],
    [7, 3, 5, 6, 0]
])

In [5]:
num_ants = 10
num_iterations = 50
evaporation_rate = 0.5
alpha = 1   # pheromone importance
beta = 2    # visibility importance

In [7]:
num_cities = len(distance_matrix)

pheromone = np.ones((num_cities, num_cities))

# Fix visibility (avoid division by zero)
visibility = np.zeros_like(distance_matrix, dtype=float)
for i in range(num_cities):
    for j in range(num_cities):
        if i != j:
            visibility[i][j] = 1 / distance_matrix[i][j]

In [9]:
for iteration in range(num_iterations):
    ant_routes = []

    for ant in range(num_ants):
        current_city = random.randint(0, num_cities - 1)
        visited = [current_city]

        while len(visited) < num_cities:
            probabilities = []

            for city in range(num_cities):
                if city not in visited:
                    tau = pheromone[current_city][city] ** alpha
                    eta = visibility[current_city][city] ** beta
                    probabilities.append((city, tau * eta))

            cities, probs = zip(*probabilities)
            probs = np.array(probs)
            probs = probs / probs.sum()

            next_city = np.random.choice(cities, p=probs)
            visited.append(next_city)
            current_city = next_city

        ant_routes.append(visited)

    # Pheromone update
    delta_pheromone = np.zeros((num_cities, num_cities))

    for route in ant_routes:
        route_length = sum(distance_matrix[route[i]][route[(i+1)%num_cities]] for i in range(num_cities))
        for i in range(num_cities):
            a = route[i]
            b = route[(i+1)%num_cities]
            delta_pheromone[a][b] += 1 / route_length
            delta_pheromone[b][a] += 1 / route_length

    pheromone = (1 - evaporation_rate) * pheromone + delta_pheromone

In [11]:
best_route = min(
    ant_routes,
    key=lambda route: sum(distance_matrix[route[i]][route[(i+1)%num_cities]] for i in range(num_cities))
)

shortest_distance = sum(
    distance_matrix[best_route[i]][best_route[(i+1)%num_cities]] for i in range(num_cities)
)

print("Best route:", best_route)
print("Shortest distance:", shortest_distance)

Best route: [3, 1, 0, 2, 4]
Shortest distance: 26
